# Paper-figure generator

This notebook regenerates **every figure** included in the workshop paper
(`paper/main.tex`) directly from the benchmark results stored under
`results/`. Running all cells top-to-bottom rebuilds the contents of
`paper/figs/` byte-for-byte (modulo the matplotlib font-cache).

Figures produced:

| File | Description | Used in |
|---|---|---|
| `agns_speedup_compact.{pdf,png}` | Single-column stacked-panel version of the speed-up bar chart | **Fig. 1, main paper** |
| `agns_speedup.{pdf,png}` | Two-column wide-panel version of the same bar chart | reference / slides |
| `wsm_time.{pdf,png}` | Wall-clock time vs accuracy on the nonlinear-equations problem | **Fig. 2, main paper** |
| `convergence.{pdf,png}` | Convergence of all 7 baselines on Rosenbrock-NLE p=5 and Chebyshev n=1000 | appendix Fig. 3 |
| `appendix_psweep.{pdf,png}` | Per-iteration convergence on Rosenbrock-NLE for p in {2,3,4,5,6,8} | appendix Fig. 4 |
| `appendix_logsumexp.{pdf,png}` | Convergence on the LIBSVM datasets a9a and mushrooms | appendix Fig. 5 |
| `appendix_matrix_inv.{pdf,png}` | Matrix-inversion count vs accuracy on nonlinear equations | appendix Fig. 6 |

The notebook is **idempotent**: running it on a different machine after
`bash scripts/run_all.sh` produces an identical figure set.

## Imports and project paths

In [ ]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")  # no display needed; we save to disk
import matplotlib.pyplot as plt

# Resolve project root from notebook location
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / ".." / "results").exists() else Path.cwd()
RESULTS = PROJECT_ROOT / "results"
FIGDIR = PROJECT_ROOT / "paper" / "figs"
FIGDIR.mkdir(parents=True, exist_ok=True)

assert RESULTS.exists(), f"results/ not found at {RESULTS}; run bash scripts/run_all.sh first"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"writing figures to {FIGDIR}")

In [ ]:
# A consistent style for every paper figure
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "savefig.bbox": "tight",
})

# Method styling, shared across every plot
METHOD_STYLES = {
    "gns_exact":          ("GNS-Exact",                "#e41a1c", "-",  1.6),
    "gns_inexact":        ("GNS-Inexact",              "#377eb8", "-",  1.6),
    "gns_wsm":            ("GNS-Fisher (WSM)",         "#377eb8", "-",  1.6),
    "super_newton":       ("Super-Newton",             "#984ea3", "-.", 1.6),
    "cubic_newton":       ("Cubic Newton",             "#a65628", "-.", 1.6),
    "agns_lookahead":     ("AGNS-Lookahead (ours)",    "#dd1c77", "-",  2.4),
    "agns_iterate":       ("AGNS-Iterate (ablation)",  "#ff7f00", ":",  1.6),
    "agns_wsm_lookahead": ("AGNS-WSM-Lookahead (ours)","#dd1c77", "-",  2.4),
    "agns_wsm_iterate":   ("AGNS-WSM-Iterate",         "#ff7f00", ":",  1.6),
    "gradient":           ("Gradient",                 "gray",    "--", 1.0),
    "fast_gradient":      ("Fast Gradient",            "black",   ":",  1.0),
}

def load_run(name):
    """Load summary.json and per-method history pickles for a result dir."""
    rdir = RESULTS / name
    with open(rdir / "summary.json") as f:
        summary = json.load(f)
    histories = {}
    for mkey in summary["methods"]:
        path = rdir / f"{mkey}_history.pkl"
        if path.exists():
            with open(path, "rb") as h:
                histories[mkey] = pickle.load(h)
    return summary, histories

def truncate_to_eps(values, eps=1e-8):
    """Return prefix ending at first index where the residual stays below eps."""
    arr = np.asarray(values)
    arr = np.clip(arr, 1e-20, None)
    cutoff = np.searchsorted(-arr, -eps) + 1
    return min(len(arr), cutoff)

def save_figure(fig, name):
    fig.tight_layout()
    for ext in ("pdf", "png"):
        out = FIGDIR / f"{name}.{ext}"
        fig.savefig(out, dpi=200)
        print(f"  wrote {out}")
    plt.close(fig)

ros_ps = (2, 3, 4, 5, 6, 8)
cheb_ns = (200, 500, 1000, 2000)

def collect(prefix, axis_values):
    rows = []
    for v in axis_values:
        s, _ = load_run(f"{prefix}{v}")
        rows.append((v, s["methods"]["gns_inexact"]["n_iters"],
                        s["methods"]["agns_lookahead"]["n_iters"]))
    return rows

ros_rows = collect("rosenbrock_nle_p", ros_ps)
cheb_rows = collect("chebyshev_n", cheb_ns)

## Figure 1 (main paper) — AGNS speed-up bar charts

Apples-to-apples comparison of GNS-Inexact and AGNS-Lookahead
iteration counts to reach the target accuracy `eps = 1e-8`. Both
methods use the same Hessian approximation and the same adaptive
search; the only differences are the Nesterov extrapolation and
gradient restart introduced in AGNS.

We generate two layouts of the same data:

* `agns_speedup_compact.{pdf,png}` — **single-column, stacked panels**, used in the workshop paper.
* `agns_speedup.{pdf,png}` — full-width, side-by-side panels (kept for slides and the README).

In [ ]:
# Wide (side-by-side) layout — used in slides and README, not in the paper.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 3.0))
for ax, rows, xlabel, title in (
    (ax1, ros_rows,  r"objective power $p$",  r"Rosenbrock-NLE $p$-sweep"),
    (ax2, cheb_rows, r"dimension $n$",        r"Chebyshev ($p=4$) dimension sweep"),
):
    xs = [r[0] for r in rows]
    gns = [r[1] for r in rows]
    agns = [r[2] for r in rows]
    x = np.arange(len(xs))
    w = 0.36
    ax.bar(x - w/2, gns, w, label="GNS-Inexact", color="#2ca25f", edgecolor="black", linewidth=0.5)
    ax.bar(x + w/2, agns, w, label="AGNS-Lookahead (ours)", color="#dd1c77", edgecolor="black", linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels([str(v) for v in xs])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"iterations to $f(x_k)-f^\star \!<\! 10^{-8}$")
    ax.set_title(title)
    ax.legend(loc="best")
save_figure(fig, "agns_speedup")

In [ ]:
# Compact (stacked) layout — **THIS is the figure used in the paper as Fig. 1**.
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "legend.fontsize": 7.5, "xtick.labelsize": 8, "ytick.labelsize": 8})
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.4, 4.6))
for ax, rows, xlabel, title in (
    (ax1, ros_rows,  r"objective power $p$",  r"Rosenbrock-NLE"),
    (ax2, cheb_rows, r"dimension $n$",        r"Chebyshev ($p=4$)"),
):
    xs = [r[0] for r in rows]
    gns = [r[1] for r in rows]
    agns = [r[2] for r in rows]
    x = np.arange(len(xs))
    w = 0.36
    ax.bar(x - w/2, gns, w, label="GNS-Inexact", color="#2ca25f", edgecolor="black", linewidth=0.5)
    ax.bar(x + w/2, agns, w, label="AGNS-Lookahead (ours)", color="#dd1c77", edgecolor="black", linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels([str(v) for v in xs])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"iters to $\varepsilon{=}10^{-8}$")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=7)
save_figure(fig, "agns_speedup_compact")

# Restore default font sizes for the remaining figures
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "legend.fontsize": 8, "xtick.labelsize": 9, "ytick.labelsize": 9})

## Appendix figure — convergence vs all baselines

Per-iteration convergence on two representative non-convex problems.
Trajectories are truncated at the first iterate satisfying the
stopping criterion; values are clipped at `1e-20` to avoid `log(0)`
in the semilog plot.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 3.0))
panels = [
    ("rosenbrock_nle", r"Rosenbrock-NLE ($p=5$)"),
    ("chebyshev",      r"Chebyshev chain ($n\!=\!1000, p\!=\!4$)"),
]

for ax, (name, title) in zip((ax1, ax2), panels):
    s, hists = load_run(name)
    f_star = s["f_star"]
    for mkey, (lbl, col, ls, lw) in METHOD_STYLES.items():
        if mkey not in hists:
            continue
        func = np.array(hists[mkey]["func"])
        resid = np.clip(func - f_star, 1e-20, None)
        n_pts = truncate_to_eps(resid)
        ax.semilogy(np.arange(n_pts), resid[:n_pts], label=lbl, color=col, linestyle=ls, linewidth=lw)
    ax.set_xlabel("iterations")
    ax.set_ylabel(r"$f(x_k) - f^\star$")
    ax.set_title(title)
    ax.legend(loc="upper right")

save_figure(fig, "convergence")

## Figure 2 (main paper) — WSM wall-clock advantage

Wall-clock time vs accuracy on the nonlinear-equations problem with
`n=100, m=200, p=4`. The Sherman–Morrison rank-1 inverse turns the
$O(m^3)$ Cholesky of GNS-Exact into an $O(m^2)$ two-solve, paying off
in two orders of magnitude of wall-clock improvement.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4.4, 3.2))
s, hists = load_run("nonlinear_equations_wsm")
f_star = s["f_star"]
for mkey in ("gns_exact", "gns_wsm", "agns_wsm_lookahead", "gradient", "fast_gradient"):
    if mkey not in hists:
        continue
    lbl, col, ls, lw = METHOD_STYLES[mkey]
    if mkey == "gns_exact":
        lbl = "GNS-Exact (CHO solve)"
    elif mkey == "gns_wsm":
        lbl = "GNS-Fisher (Sherman-Morrison)"
    func = np.array(hists[mkey]["func"])
    time = np.array(hists[mkey]["time"])
    resid = np.clip(func - f_star, 1e-20, None)
    n_pts = truncate_to_eps(resid)
    ax.semilogy(time[:n_pts], resid[:n_pts], label=lbl, color=col, linestyle=ls, linewidth=lw)
ax.set_xlabel("wall-clock time (s)")
ax.set_ylabel(r"$f(x_k) - f^\star$")
ax.set_title(r"Nonlinear equations  ($n\!=\!100, m\!=\!200, p\!=\!4$)")
ax.set_xscale("log")
ax.legend(loc="lower left", fontsize=7)
save_figure(fig, "wsm_time")

## Appendix figure — Rosenbrock-NLE p-sweep convergence

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.4), sharey=True)
for ax, p in zip(axes.flat, ros_ps):
    s, hists = load_run(f"rosenbrock_nle_p{p}")
    f_star = s["f_star"]
    for mkey, (lbl, col, ls, lw) in METHOD_STYLES.items():
        if mkey not in hists:
            continue
        func = np.array(hists[mkey]["func"])
        resid = np.clip(func - f_star, 1e-20, None)
        n_pts = truncate_to_eps(resid)
        # Trim long gradient-method tails to keep panels readable
        n_pts = min(n_pts, 200)
        ax.semilogy(np.arange(n_pts), resid[:n_pts], label=lbl, color=col, linestyle=ls, linewidth=lw)
    ax.set_title(rf"Rosenbrock-NLE  $p={p}$")
    ax.set_xlabel("iterations")
    if ax in axes[:, 0]:
        ax.set_ylabel(r"$f(x_k) - f^\star$")

# One shared legend below the grid
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), fontsize=8)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_figure(fig, "appendix_psweep")

## Appendix figure — LogSumExp convergence on real LIBSVM data

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 3.0))
for ax, (name, title) in zip(
        (ax1, ax2),
        (("logsumexp_real_a9a",       r"LogSumExp on a9a ($\mu=0.1$)"),
         ("logsumexp_real_mushrooms", r"LogSumExp on mushrooms ($\mu=0.1$)"))):
    s, hists = load_run(name)
    f_star = s["f_star"]
    for mkey, (lbl, col, ls, lw) in METHOD_STYLES.items():
        if mkey not in hists:
            continue
        func = np.array(hists[mkey]["func"])
        resid = np.clip(func - f_star, 1e-20, None)
        n_pts = truncate_to_eps(resid)
        ax.semilogy(np.arange(n_pts), resid[:n_pts], label=lbl, color=col, linestyle=ls, linewidth=lw)
    ax.set_xlabel("iterations")
    ax.set_ylabel(r"$f(x_k) - f^\star$")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=7)
save_figure(fig, "appendix_logsumexp")

## Appendix figure — matrix-inversion count vs accuracy

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4.4, 3.2))
s, hists = load_run("nonlinear_equations_wsm")
f_star = s["f_star"]
for mkey in ("gns_exact", "gns_wsm", "agns_wsm_lookahead", "agns_wsm_iterate"):
    if mkey not in hists:
        continue
    lbl, col, ls, lw = METHOD_STYLES[mkey]
    func = np.array(hists[mkey]["func"])
    invs = np.array(hists[mkey]["matrix_inverses"])
    resid = np.clip(func - f_star, 1e-20, None)
    n_pts = truncate_to_eps(resid)
    ax.semilogy(invs[:n_pts], resid[:n_pts], label=lbl, color=col, linestyle=ls, linewidth=lw)
ax.set_xlabel("matrix solves")
ax.set_ylabel(r"$f(x_k) - f^\star$")
ax.set_title(r"Nonlinear equations  ($n\!=\!100, m\!=\!200, p\!=\!4$)")
ax.legend(loc="upper right", fontsize=8)
save_figure(fig, "appendix_matrix_inv")

## All figures regenerated

The cell below verifies that every file referenced by `paper/main.tex`
now exists in `paper/figs/`. The check is exhaustive: it parses
`paper/main.tex` for `\includegraphics{figs/...}` references and
compares against the actual contents of `paper/figs/`.

In [ ]:
import re

main_tex = (PROJECT_ROOT / "paper" / "main.tex").read_text()
needed = sorted({Path(m).name for m in re.findall(r"figs/[A-Za-z0-9_]+\.pdf", main_tex)})
print("Figures referenced by paper/main.tex:")
for n in needed: print(f"  {n}")

missing = [n for n in needed if not (FIGDIR / n).exists()]
if missing:
    print("\nFAIL — missing figures:")
    for n in missing: print(f"  {n}")
    raise AssertionError(f"{len(missing)} figure(s) missing")
else:
    print(f"\nOK: all {len(needed)} paper-referenced figures are present in {FIGDIR}")

# Also list extras (figures we generate but the paper doesn't reference)
produced = sorted({p.name for p in FIGDIR.glob("*.pdf")})
extras = [p for p in produced if p not in needed]
if extras:
    print("\nAdditional figures generated (not used in paper, kept for reference):")
    for e in extras: print(f"  {e}")